# Lab 5: SFT Real com TRL

## Lab 5: Supervised Fine-Tuning real com TRL

**Nota sobre o modelo:** testamos primeiro com `SmolLM2-135M` (modelo real,
que entende linguagem) e cada step de treino levou **40-75 segundos** neste
CPU — inviável pra um lab que você quer rodar e iterar. Trocamos pro
`sshleifer/tiny-gpt2` (~100k parâmetros): o `SFTTrainer` é o mesmo usado em
produção, a mecânica é idêntica — só o modelo é pequeno o suficiente pra
treinar em segundos, não minutos. Com um modelo real numa GPU, é o mesmo
código.

In [1]:
!pip install -q transformers torch trl datasets

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig

### 1. Dataset real (reaproveitando o Lab 4)

In [2]:
raw = load_dataset("databricks/databricks-dolly-15k", split="train[:40]")
raw = raw.filter(lambda ex: ex["instruction"] and ex["response"] and not ex["context"])
print(f"✓ {len(raw)} exemplos reais (sem contexto extra, pra manter os prompts curtos)")
print(raw[0])

✓ 27 exemplos reais (sem contexto extra, pra manter os prompts curtos)
{'instruction': 'Which is a species of fish? Tope or Rope', 'context': '', 'response': 'Tope', 'category': 'classification'}


### 2. Formatando em texto simples (tiny-gpt2 não tem chat template)

In [3]:
tokenizer = AutoTokenizer.from_pretrained("sshleifer/tiny-gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(ex):
    return {"text": f"Instruction: {ex['instruction']}\nResponse: {ex['response']}{tokenizer.eos_token}"}

dataset = raw.map(format_example)
print(dataset[0]["text"][:200])

Instruction: Which is a species of fish? Tope or Rope
Response: Tope<|endoftext|>


### 3. Modelo ANTES do fine-tuning — baseline

In [4]:
import torch

model = AutoModelForCausalLM.from_pretrained("sshleifer/tiny-gpt2")

def generate(model, prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_prompt = "Instruction: What is the capital of France?\nResponse:"
print("ANTES do fine-tuning:")
print(generate(model, test_prompt))

ANTES do fine-tuning:
Instruction: What is the capital of France?
Response: stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs


**Resultado esperado:** texto essencialmente aleatório/sem sentido — pesos
não-treinados do `tiny-gpt2` não produzem respostas coerentes.

### 4. Fine-tuning real com SFTTrainer (biblioteca de produção)

In [5]:
sft_config = SFTConfig(
    output_dir="./sft_output",
    per_device_train_batch_size=4,
    max_steps=30,
    learning_rate=5e-3,
    logging_steps=5,
    report_to="none",
    bf16=False, fp16=False,
    max_length=64,
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=dataset)
trainer.train()
print("\n✓ Fine-tuning concluído")

{'loss': '10.55', 'grad_norm': '0.4608', 'learning_rate': '0.004333', 'entropy': '10.82', 'num_tokens': '855', 'mean_token_accuracy': '0', 'epoch': '0.7143'}
{'loss': '10.57', 'grad_norm': '0.1907', 'learning_rate': '0.0035', 'entropy': '10.82', 'num_tokens': '1729', 'mean_token_accuracy': '0', 'epoch': '1.429'}
{'loss': '10.5', 'grad_norm': '0.2812', 'learning_rate': '0.002667', 'entropy': '10.82', 'num_tokens': '2514', 'mean_token_accuracy': '0.00673', 'epoch': '2.143'}
{'loss': '10.5', 'grad_norm': '0.4779', 'learning_rate': '0.001833', 'entropy': '10.82', 'num_tokens': '3417', 'mean_token_accuracy': '0.009269', 'epoch': '2.857'}
{'loss': '10.44', 'grad_norm': '0.2428', 'learning_rate': '0.001', 'entropy': '10.82', 'num_tokens': '4183', 'mean_token_accuracy': '0.0258', 'epoch': '3.571'}
{'loss': '10.46', 'grad_norm': '0.2091', 'learning_rate': '0.0001667', 'entropy': '10.82', 'num_tokens': '5095', 'mean_token_accuracy': '0.02159', 'epoch': '4.286'}
{'train_runtime': '10.62', 'train_

**Resultado esperado:** logs de loss a cada 5 steps. A queda aqui é bem
mais sutil que no Lab 3 (que treinava em 3 frases repetidas — fácil de
decorar). Aqui são 27 exemplos **diferentes** do dataset real — pedir pra
um modelo de 100k parâmetros generalizar um padrão em só 30 steps sobre
dado diverso é pedir muito; o loss deve cair pouco (algo como 10.55 →
10.50). Isso não é falha do código — é a Semana 5.4 na prática: modelos
pequenos, poucos steps e datasets diversos convergem devagar. Com
`SmolLM2` numa GPU real e mais steps, a queda seria bem mais clara.

### 5. Modelo DEPOIS do fine-tuning — comparação

In [6]:
print("DEPOIS do fine-tuning:")
print(generate(trainer.model, test_prompt))

print("\n📊 Comparação lado a lado:")
print(f"ANTES:  {generate(AutoModelForCausalLM.from_pretrained('sshleifer/tiny-gpt2'), test_prompt)}")
print(f"DEPOIS: {generate(trainer.model, test_prompt)}")

DEPOIS do fine-tuning:
Instruction: What is the capital of France?
Response:ructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionruction

📊 Comparação lado a lado:
ANTES:  Instruction: What is the capital of France?
Response: stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs
DEPOIS: Instruction: What is the capital of France?
Response:ructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionructionruction


**Resultado esperado:** as duas saídas são texto sem sentido factual (o
modelo é pequeno e mal-treinado demais pra "saber" a capital da França) —
mas devem ser **diferentes uma da outra**. No nosso teste, ANTES repetia
"stairs" e DEPOIS degenerou repetindo um fragmento de "Instruction"
(provavelmente um token que ficou super-reforçado nos 30 steps de treino
sobre um dataset onde "Instruction:" aparece em toda linha). Isso não é o
resultado "bonito" que você quer em produção — é a prova de que os pesos
mudaram de verdade com o treino, o que é exatamente o que este lab
existe pra demonstrar. Um modelo maior, com mais steps, convergiria pra
respostas coerentes em vez de repetição degenerada.

**Próximos passos:** Semana 6 mostra como fazer esse mesmo fine-tuning
treinando só ~1% dos parâmetros (LoRA) em vez do modelo inteiro.